# 02 - Pre-anotacao por regras

A anotacao manual de mil titulos do zero e cara. A estrategia adotada aqui e
gerar um rascunho automatico com dicionarios e expressoes regulares, importar
esse rascunho no Doccano e fazer apenas a **revisao** das entidades.

O mesmo componente tem um segundo uso: ele e o baseline da comparacao
experimental, ja que representa a solucao sem aprendizado de maquina.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import collections
import re

import pandas as pd

from src.dados import (
    carregar_produtos,
    titulos_unicos,
    marcas_do_catalogo,
    salvar_jsonl,
    carregar_jsonl,
    DIR_ANOT,
)

BASE = "cellphone"

from src.preanotacao import anotar_base, anotar_titulo, _regex_marcas

In [2]:
produtos = titulos_unicos(carregar_produtos(BASE))
marcas = marcas_do_catalogo(produtos)
print(f"titulos: {len(produtos)} | marcas no gazetteer: {len(marcas)}")

titulos: 1032 | marcas no gazetteer: 60


## 1. Esquema de tags

| Tag | Descricao | Exemplo |
|---|---|---|
| TIPO | categoria funcional do produto | Smartphone, Capa, Pelicula |
| MARCA | fabricante | Samsung, Apple, Multilaser |
| MODELO | linha/modelo do produto principal | iPhone 14 Pro Max, Galaxy A53 |
| COMPATIBILIDADE | modelo a que um acessorio se destina | (Capa para) iPhone 13 Pro |
| MEMORIA | armazenamento interno | 128GB |
| RAM | memoria de trabalho | 4GB RAM |
| COR | cor declarada | Preto, Azul Pacifico |
| TELA | tamanho da tela | Tela 6,5 |
| REDE | geracao de rede movel | 4G, 5G |
| CAMERA | resolucao da camera | 13MP |
| BATERIA | capacidade da bateria | 5000mAh |
| POTENCIA | potencia ou voltagem | 20W, Bivolt |
| CODIGO | codigo do fabricante | MM2Y3ZE/A, NK041 |

As regras de desempate estao em `docs/guia-anotacao.md`.

## 2. Aplicacao das regras

In [3]:
preanotado = anotar_base(produtos, marcas)

contagem = collections.Counter(tag for r in preanotado for _, _, tag in r["entities"])
sem_entidade = sum(1 for r in preanotado if not r["entities"])

print(f"entidades geradas: {sum(contagem.values())}")
print(f"titulos sem nenhuma entidade: {sem_entidade}")
pd.Series(dict(contagem.most_common())).to_frame("ocorrencias")

entidades geradas: 5786
titulos sem nenhuma entidade: 1


,ocorrencias
MARCA,1016
TIPO,862
COR,664
MEMORIA,614
MODELO,507
CODIGO,464
TELA,434
CAMERA,341
RAM,295
COMPATIBILIDADE,286


## 3. Inspecao qualitativa

Amostra aleatoria para conferir o comportamento das regras antes de exportar.

In [5]:
import random

random.seed(42)
for registro in random.sample(preanotado, 15):
    print(registro["text"])
    marcados = [(registro["text"][i:f], tag) for i, f, tag in registro["entities"]]
    print("   ", marcados)
    print()

Capa iPhone 11 Pro Max Apple, Couro Azul - MX0G2ZM/A
    [('Capa', 'TIPO'), ('iPhone 11 Pro Max', 'COMPATIBILIDADE'), ('Apple', 'MARCA'), ('Azul', 'COR'), ('MX0G2ZM/A', 'CODIGO')]

Apple iPhone 14 Pro 512GB Preto-espacial
    [('Apple', 'MARCA'), ('iPhone 14 Pro', 'MODELO'), ('512GB', 'MEMORIA'), ('Preto-espacial', 'COR')]

Smartphone Ms50 4G Multilaser Câmera 8 Mp + 5 Mp Quad Core 1Gb Ram Branco - P9014
    [('Smartphone', 'TIPO'), ('Ms50', 'MODELO'), ('4G', 'REDE'), ('Multilaser', 'MARCA'), ('8 Mp', 'CAMERA'), ('5 Mp', 'CAMERA'), ('1Gb Ram', 'RAM'), ('Branco', 'COR'), ('P9014', 'CODIGO')]

Smartphone Ms45S Sênior 3G Tela 4,5 Pol. Dual Câmera 5.0Mp + 3.0Mp Dual Chip Android 6.0 Fm - NB745
    [('Smartphone', 'TIPO'), ('Ms45S', 'MODELO'), ('3G', 'REDE'), ('Tela 4,5 Pol', 'TELA'), ('5.0Mp', 'CAMERA'), ('3.0Mp', 'CAMERA'), ('NB745', 'CODIGO')]

Kit Smartphone Samsung Galaxy A13 128GB Preto + Headphone Bluetooth Preto | GT
    [('Kit', 'TIPO'), ('Samsung', 'MARCA'), ('Galaxy A13', 'MODELO

## 4. Limitacoes observadas

Erros que a revisao manual no Doccano precisa corrigir:

1. Modelos de marcas sem padrao mapeado ficam de fora (`LG K8 Plus`,
   `Obasmart Conecta`).
2. Em acessorios cujo titulo cita o tipo do aparelho alvo
   (`Cinta Esportiva ... Smartphone de Ate 5.5 Pol`), o termo e marcado como TIPO
   quando na verdade e compatibilidade.
3. Codigos curtos do fabricante podem ser confundidos com modelo (`S532`).

Esses casos justificam a etapa de revisao e aparecem depois como diferenca entre
o baseline de regras e os modelos treinados.

## 5. Exportacao para o Doccano

O arquivo gerado aqui e o rascunho. Apos a revisao no Doccano, o export volta
para `data/annotations/cellphone.ibyte.jsonl`, que e a base de referencia usada
no treinamento e na avaliacao.

In [6]:
destino = DIR_ANOT / f"{BASE}.ibyte.preanotado.jsonl"
salvar_jsonl(preanotado, destino)
print(f"salvo em: {destino}")
print(f"linhas: {len(preanotado)}")

salvo em: /Users/mpes/Desktop/ner-nivson/data/annotations/cellphone.ibyte.preanotado.jsonl
linhas: 1032


## 6. Amostra para revisao manual

A revisao e feita por uma pessoa so, entao anotar os 1032 titulos seria caro sem
ganho proporcional. A opcao adotada e revisar uma amostra de 400 titulos
estratificada por categoria, preservando a proporcao entre produto principal e
acessorios. O restante fica disponivel como conjunto nao anotado, util para
inspecao qualitativa das predicoes dos modelos.

O tamanho da amostra e revisitado no notebook de avaliacao, pela curva de
aprendizado: se o desempenho ainda estiver subindo em 400 exemplos, vale anotar
mais.

In [7]:
import random

TAMANHO_AMOSTRA = 400

random.seed(42)
por_categoria = collections.defaultdict(list)
for indice, produto in enumerate(produtos):
    por_categoria[produto["categoria"] or "sem categoria"].append(indice)

indices_amostra = []
for categoria, indices in por_categoria.items():
    cota = round(len(indices) * TAMANHO_AMOSTRA / len(produtos))
    cota = max(1, min(cota, len(indices)))
    indices_amostra.extend(random.sample(indices, cota))

indices_amostra = sorted(indices_amostra)
restantes = [i for i in range(len(produtos)) if i not in set(indices_amostra)]

resumo = pd.DataFrame([
    {
        "categoria": categoria,
        "na_base": len(indices),
        "na_amostra": sum(1 for i in indices_amostra if i in set(indices)),
    }
    for categoria, indices in sorted(por_categoria.items(), key=lambda x: -len(x[1]))
])
print(f"amostra: {len(indices_amostra)} titulos | restante: {len(restantes)}")
resumo

amostra: 401 titulos | restante: 631


,categoria,na_base,na_amostra
0,Smartphone,659,255
1,Capa para Celular,207,80
2,Acessórios para Celular,43,17
3,Película para Celular,42,16
4,Carregadores,23,9
5,Celular,21,8
6,Celular Simples,17,7
7,Apoio para Smartphone,8,3
8,Suporte para Celular,7,3
9,Carregador de Celular,2,1


In [8]:
salvar_jsonl([preanotado[i] for i in indices_amostra], DIR_ANOT / f"{BASE}.ibyte.revisar.jsonl")
salvar_jsonl([preanotado[i] for i in restantes], DIR_ANOT / f"{BASE}.ibyte.restante.jsonl")
print("arquivos gerados em data/annotations/")

arquivos gerados em data/annotations/


## Proximo passo

Importar `cellphone.ibyte.revisar.jsonl` no Doccano, revisar as entidades
seguindo `docs/guia-anotacao.md` e exportar o resultado como
`data/annotations/cellphone.ibyte.jsonl`. Esse arquivo e a referencia usada no
treinamento e na avaliacao.